In [1]:
# Tarea
# INSTRUCCIONES:
# 1. Procesa todas las reseñas usando limpiar_lematizar(...)
# 2. Junta todos los tokens en una sola lista
# 3. Crea un BigramCollocationFinder con esa lista
# 4. Aplica un filtro mínimo de frecuencia
# 5. Muestra:
#    - los 5 bigramas más frecuentes
#    - los 5 bigramas con mayor PMI
# 6. Responde:
#    a) ¿Qué problemas aparecen más por frecuencia?
#    b) ¿Qué conceptos específicos aparecen con PMI?
#    c) ¿Por qué no siempre coinciden frecuencia y PMI?

dataset_vuelos_alumnos = [
    "El vuelo fue retrasado por 3 horas, pésimo servicio.",
    "La comida estaba fría y el asiento roto.",
    "El vuelo retrasado me hizo perder la conexión en Madrid.",
    "Excelente atención de la azafata, pero el asiento roto fue muy incómodo.",
    "Mi vuelo fue cancelado y el personal de tierra fue grosero.",
    "El vuelo retrasado es una falta de respeto al pasajero.",
    "Me cobraron doble por el equipaje extra. Pésimo servicio en mostrador.",
    "El asiento roto me lastimó la espalda, exijo reembolso del vuelo.",
    "Llamé a servicio al cliente por el equipaje extra y me colgaron.",
    "El vuelo cancelado arruinó mis vacaciones.",
    "Mi vuelo fue cancelado y nadie me dio solución.",
    "Pésimo servicio en mostrador y equipaje extra demasiado caro.",
    "El asiento roto fue muy incómodo durante todo el vuelo."
]

# ESCRIBE TU CÓDIGO AQUÍ ABAJO

# Librerías

In [3]:
# Instalación y carga de archivos

# Objetivo de esta celda:
# Preparar el entorno para trabajar con:
# 1. spaCy: análisis lingüístico (lematización y POS tagging)
# 2. NLTK: n-gramas, colocaciones y PMI

import sys
import subprocess

# --- SECCIÓN 1: Gestión de spaCy ---
# Intentamos importar la librería spaCy. 
# Si no está instalada, usamos subprocess para ejecutar el comando pip install automáticamente.
try:
    import spacy
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])
    import spacy

# --- SECCIÓN 2: Carga del modelo de lenguaje ---
# spaCy necesita un modelo del idioma a trabajar.
# "es_core_news_sm" es un modelo pequeño de español que ya sabe hacer tareas como tokenización, lematización y POS tagging.
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    # Si el modelo no ha sido descargado previamente, lo descargamos mediante la terminal
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "es_core_news_sm"])
    nlp = spacy.load('es_core_news_sm')

# --- SECCIÓN 3: Herramientas NLTK y Colecciones ---
import nltk

# Importamos las utilidades necesarias para procesar el texto:
# - ngrams: Para generar secuencias de N palabras.
# - BigramCollocationFinder: Para identificar pares de palabras que suelen ir juntos.
# - BigramAssocMeasures: Provee métricas estadísticas como PMI para evaluar colocalizaciones.
from nltk.util import ngrams
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures

# Counter nos servirá para realizar conteos de frecuencia de forma eficiente.
from collections import Counter

1. **Procesa todas las reseñas usando limpiar_lematizar.**

In [5]:
# Función de limpieza y lematización

def limpiar_lematizar(texto, verbose=0):
    """
    Realiza un preprocesamiento avanzado: convierte a minúsculas, elimina ruido 
    (números/puntuación), filtra por categorías gramaticales y aplica lematización.
    """

    if verbose: print(f"[0] Texto original: {texto}")
    # Procesamos el texto con el modelo de spaCy (nlp) y lo convertimos a minúsculas
    documento = nlp(texto.lower())
    if verbose: print(f"[1] Texto en minúsculas y procesado por spaCy")
    
    tokens_limpios = []

    # Definimos qué categorías gramaticales (POS tags) aportan significado relevante
    # NOUN: Sustantivos, ADJ: Adjetivos, VERB: Verbos
    etiquetas_validas = {"NOUN", "ADJ", "VERB"}
    
    # Definimos términos de negación para no perder el sentido crítico en el análisis
    negaciones = {"no", "ni", "nunca"}

    # Recorremos cada token (palabra o signo) identificado por spaCy
    for token in documento:
        # Filtramos para que solo pasen caracteres alfabéticos (evita "3", "!", etc.)
        if token.is_alpha:
            
            # Prioridad 1: Si es una palabra de negación, la guardamos literal. Esto es vital para análisis de sentimientos (ej. "no funciona")
            if token.text in negaciones:
                tokens_limpios.append(token.text)
                if verbose: print(f" - Negación detectada: {token.text}")
            
            # Prioridad 2: Si es sustantivo, adjetivo o verbo, guardamos su LEMA. 
            # El lema es la raíz de la palabra (ej. "vuelos" -> "vuelo", "retrasado" -> "retrasar")
            elif token.pos_ in etiquetas_validas:
                tokens_limpios.append(token.lemma_)
                if verbose: print(f" - Palabra válida ({token.pos_}): {token.text} -> Lema: {token.lemma_}")
                
    # Retornamos la lista de palabras normalizadas y filtradas
    if verbose: print(f"[Resultado Final]: {tokens_limpios}\n")
    return tokens_limpios

Revisamos que funcione la función:

In [7]:
print(limpiar_lematizar("El vuelo fue retrasado por 3 horas, pésimo servicio.",1))

[0] Texto original: El vuelo fue retrasado por 3 horas, pésimo servicio.
[1] Texto en minúsculas y procesado por spaCy
 - Palabra válida (NOUN): vuelo -> Lema: vuelo
 - Palabra válida (VERB): retrasado -> Lema: retrasar
 - Palabra válida (NOUN): horas -> Lema: hora
 - Palabra válida (ADJ): pésimo -> Lema: pésimo
 - Palabra válida (NOUN): servicio -> Lema: servicio
[Resultado Final]: ['vuelo', 'retrasar', 'hora', 'pésimo', 'servicio']

['vuelo', 'retrasar', 'hora', 'pésimo', 'servicio']


2. **Junta todos los tokens en una sola lista**

In [9]:
# Generamos una lista de listas: aplicamos la función 'limpiar_lematizar' a cada reseña del dataset.
# Esto nos da como resultado los tokens filtrados y lematizados de forma individual por cada comentario.
tokens_por_resena = [limpiar_lematizar(resena) for resena in dataset_vuelos_alumnos]
tokens_por_resena

[['vuelo', 'retrasar', 'hora', 'pésimo', 'servicio'],
 ['comida', 'frío', 'asiento', 'roto'],
 ['vuelo', 'retrasado', 'hacer', 'perder', 'conexión'],
 ['excelente', 'atención', 'azafata', 'asiento', 'roto', 'incómodo'],
 ['vuelo', 'cancelar', 'personal', 'tierra', 'grosero'],
 ['vuelo', 'retrasado', 'falta', 'respeto', 'pasajero'],
 ['cobrar', 'doble', 'equipaje', 'extra', 'pésimo', 'servicio', 'mostrador'],
 ['asiento', 'roto', 'lastimar', 'espalda', 'exijo', 'reembolso', 'vuelo'],
 ['llamar', 'servicio', 'cliente', 'equipaje', 'extra', 'colgar'],
 ['vuelo', 'cancelado', 'arruinar', 'vacación'],
 ['vuelo', 'cancelar', 'dar', 'solución'],
 ['pésimo', 'servicio', 'mostrador', 'equipaje', 'extra', 'caro'],
 ['asiento', 'roto', 'incómodo', 'vuelo']]

In [10]:
# Aplanamos la lista de listas (flattening) para obtener una sola lista con todas las palabras del dataset.
# Esto es útil para realizar análisis globales, como contar las palabras más frecuentes en todas las quejas.
todos_los_tokens = [t for sub in tokens_por_resena for t in sub]
print(f"""
Tamaño de lista de tokens: {len(todos_los_tokens)} \n
Todos los tokens:{todos_los_tokens}""")


Tamaño de lista de tokens: 68 

Todos los tokens:['vuelo', 'retrasar', 'hora', 'pésimo', 'servicio', 'comida', 'frío', 'asiento', 'roto', 'vuelo', 'retrasado', 'hacer', 'perder', 'conexión', 'excelente', 'atención', 'azafata', 'asiento', 'roto', 'incómodo', 'vuelo', 'cancelar', 'personal', 'tierra', 'grosero', 'vuelo', 'retrasado', 'falta', 'respeto', 'pasajero', 'cobrar', 'doble', 'equipaje', 'extra', 'pésimo', 'servicio', 'mostrador', 'asiento', 'roto', 'lastimar', 'espalda', 'exijo', 'reembolso', 'vuelo', 'llamar', 'servicio', 'cliente', 'equipaje', 'extra', 'colgar', 'vuelo', 'cancelado', 'arruinar', 'vacación', 'vuelo', 'cancelar', 'dar', 'solución', 'pésimo', 'servicio', 'mostrador', 'equipaje', 'extra', 'caro', 'asiento', 'roto', 'incómodo', 'vuelo']


3. **Crea un <code> BigramCollocationFinder </code>**

In [12]:
buscador = BigramCollocationFinder.from_words(todos_los_tokens)
buscador

In [13]:
# Para saber qué está haciendo, podemos ver cuántos bigramas encontró en total
print(f"El buscador ha identificado {len(list(buscador.ngram_fd.items()))} bigramas únicos.")

El buscador ha identificado 55 bigramas únicos.


4. **Aplica un filtro mínimo de frecuencia**

In [15]:
# (Por ejemplo, que el bigrama aparezca al menos 3 o 5 veces para evitar ruido)
buscador.apply_freq_filter(3) 

In [16]:
# Para saber qué está haciendo, podemos ver cuántos bigramas encontró en total
print(f"El buscador ha identificado {len(list(buscador.ngram_fd.items()))} bigramas únicos.")

El buscador ha identificado 3 bigramas únicos.


5. **Muestra los resultados**

In [18]:
metricas = BigramAssocMeasures()
metricas

In [19]:
print("\nTop bigramas por frecuencia")
print(buscador.score_ngrams(metricas.raw_freq)[:5])

print("\n Top bigramas por PMI")
print(buscador.score_ngrams(metricas.pmi)[:5])


Top bigramas por frecuencia
[(('asiento', 'roto'), 0.058823529411764705), (('equipaje', 'extra'), 0.04411764705882353), (('pésimo', 'servicio'), 0.04411764705882353)]

 Top bigramas por PMI
[(('equipaje', 'extra'), 4.502500340529183), (('asiento', 'roto'), 4.087462841250339), (('pésimo', 'servicio'), 4.087462841250339)]


6. **Responde**

**a) ¿Qué problemas aparecen más por frecuencia?**


En este caso, la **frecuencia** destaca los problemas más recurrentes o comunes reportados por los usuarios.
* **El estado físico:** El problema del **('asiento', 'roto')** es el más frecuente con una probabilidad de $0.058$. 
* **Cargos o logística:** Le sigue el **('equipaje', 'extra')**.
* **Experiencia general:** El **('pésimo', 'servicio')** aparece como una queja constante.


El "problema" de la frecuencia aquí es que tiende a inflar combinaciones de palabras que son comunes de forma individual en el lenguaje de quejas, aunque no sean necesariamente las más informativas o únicas.


**b) ¿Qué conceptos específicos aparecen con PMI?**

El **PMI (Pointwise Mutual Information)** mide la fuerza de asociación entre dos palabras. Aquí, el orden cambia y prioriza la **especificidad**:
* **Logística de carga:** El concepto de **('equipaje', 'extra')** sube al primer puesto ($4.50$). Esto indica que cuando aparece la palabra "equipaje", es extremadamente probable que aparezca "extra" al lado; forman una unidad conceptual muy sólida.
  
* **Mantenimiento y Atención:** Los otros dos conceptos mantienen una asociación fuerte, pero menor que la del equipaje, sugiriendo que "asiento" o "servicio" podrían aparecer ocasionalmente con otros adjetivos, mientras que "equipaje extra" es casi un término técnico inseparable.

**c) ¿Por qué no siempre coinciden frecuencia y PMI?**

La falta de coincidencia (como vemos en el cambio de orden entre el primer y segundo lugar) se debe a cómo calculan la importancia:

1.  **La Frecuencia es absoluta:** Solo cuenta cuántas veces aparecen juntos. El "asiento roto" es el líder porque es la queja que más veces se repite en total.
2.  **El PMI es relativo:** Penaliza a las palabras que son muy comunes por separado. 
    * Si la palabra "asiento" aparece muchísimas veces en el corpus (en otros contextos como "asiento cómodo", "mi asiento", etc.), el PMI del bigrama **('asiento', 'roto')** baja un poco porque la aparición de "asiento" no garantiza siempre la de "roto".
    * Si "equipaje" y "extra" aparecen casi exclusivamente juntos, su **PMI será mayor** aunque se mencionen menos veces en total que los asientos.



**En resumen:** La frecuencia te dice qué es lo que más ocurre, mientras que el PMI te dice qué palabras tienen un "vínculo" más fuerte y único entre sí.